# Experiment 8 – Transfer Learning Using a Pre-trained Keras Model

### Deep Learning Laboratory

**Aim:** To implement Transfer Learning for image classification using a pre-trained model available in Keras.

### Learning Objectives
- Understand the use of pre-trained models in Keras.
- Load a pre-trained CNN model with ImageNet weights.
- Reuse the learned features of the model.
- Add a new classification layer for a different image-classification task.
- Evaluate the performance of the transfer-learning model.


## 1. Theory

A **pre-trained model** is a neural network that has already been trained on a large dataset. Keras provides several pre-trained CNN models such as **VGG16, VGG19, ResNet50, MobileNet, MobileNetV2, InceptionV3 and Xception**.

In this experiment, we use **VGG16**, a popular CNN model pre-trained on the ImageNet dataset.

Instead of training VGG16 from the beginning, we reuse its learned image features and replace its original classifier with a new classifier for our task.

### Architecture

```text
Input Image
     ↓
Pre-trained VGG16
     ↓
Learned Image Features
     ↓
Global Average Pooling
     ↓
New Dense Classifier
     ↓
Cat / Dog
```

### Difference from Experiment 7

- **Experiment 7:** Demonstrates the general concept of Transfer Learning using MobileNetV2.
- **Experiment 8:** Specifically demonstrates using a **pre-trained Keras model (VGG16)** for Transfer Learning.

In [ ]:
# Step 1: Import libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

## 2. Load the Dataset

We use the **Cats vs Dogs** dataset from TensorFlow Datasets. The task is binary image classification.

`0 = Cat` and `1 = Dog`.

In [ ]:
# Step 2: Install and import TensorFlow Datasets
!pip -q install tensorflow-datasets

import tensorflow_datasets as tfds

In [ ]:
# Step 3: Load a manageable subset of the Cats vs Dogs dataset
(ds_train, ds_test), ds_info = tfds.load(
    "cats_vs_dogs",
    split=["train[:80%]", "train[80%:90%]"],
    as_supervised=True,
    with_info=True
)

print("Dataset loaded successfully.")
print("Number of classes:", ds_info.features["label"].num_classes)

## 3. Image Preprocessing

VGG16 accepts RGB images. We resize all images to **224 × 224 × 3**, which is the standard input size used by VGG16.

We use the VGG16-specific `preprocess_input()` function before passing images to the pre-trained model.

In [ ]:
# Step 4: Preprocess the images
IMG_SIZE = (224, 224)
BATCH_SIZE = 32


def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label

train_ds = ds_train.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = ds_test.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)

In [ ]:
# Step 5: Display sample images
plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8))
        plt.title("Dog" if int(labels[i]) == 1 else "Cat")
        plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Load the Pre-trained VGG16 Model

`weights="imagenet"` loads weights learned from ImageNet.

`include_top=False` removes VGG16's original ImageNet classification layers. We will add our own classifier for Cats vs Dogs.

The VGG16 convolutional base is initially frozen.

In [ ]:
# Step 6: Load VGG16 with ImageNet weights
base_model = keras.applications.VGG16(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

print("VGG16 loaded successfully.")
print("Base model trainable:", base_model.trainable)

In [ ]:
# Step 7: Build the Transfer Learning model
inputs = keras.Input(shape=(224, 224, 3))

x = keras.applications.vgg16.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 5. Train the New Classifier

Only the newly added classification layer is trained. The VGG16 feature-extraction layers remain frozen.

In [ ]:
# Step 8: Train the transfer-learning model
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=3
)

In [ ]:
# Step 9: Evaluate the model
loss, accuracy = model.evaluate(test_ds, verbose=0)

print("Test Loss:", round(float(loss), 4))
print("Test Accuracy:", round(float(accuracy * 100), 2), "%")

In [ ]:
# Step 10: Plot training and validation accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("VGG16 Transfer Learning Accuracy")
plt.legend()
plt.show()

In [ ]:
# Step 11: Test predictions on sample images
for images, labels in test_ds.take(1):
    probabilities = model.predict(images, verbose=0).flatten()

    plt.figure(figsize=(10, 6))
    for i in range(min(8, len(images))):
        predicted = "Dog" if probabilities[i] >= 0.5 else "Cat"
        actual = "Dog" if int(labels[i]) == 1 else "Cat"

        plt.subplot(2, 4, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8))
        plt.title(f"Actual: {actual}\nPredicted: {predicted}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

## 6. Optional Fine-Tuning

After the classifier has been trained, we can unfreeze some of the later VGG16 layers and continue training with a very small learning rate.

This allows the model to slightly adapt its learned features to the Cats vs Dogs dataset.

In [ ]:
# Step 12: Fine-tune the last few VGG16 layers
base_model.trainable = True

# Freeze most layers; fine-tune only the last 4 layers
for layer in base_model.layers[:-4]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

fine_tune_history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=2
)

fine_loss, fine_accuracy = model.evaluate(test_ds, verbose=0)
print("Fine-tuned Test Accuracy:", round(float(fine_accuracy * 100), 2), "%")

## 7. Student Practice

Try the following modifications:

1. Change the dropout rate from `0.2` to `0.4`.
2. Change the learning rate from `0.001` to `0.0005`.
3. Fine-tune the last 2, 4 or 8 VGG16 layers.
4. Increase the number of epochs.
5. Compare the accuracy before and after fine-tuning.

Record your observations in the following format:

| Configuration | Test Accuracy |
|---|---|
| VGG16 + New Classifier | ______ |
| VGG16 + Fine-Tuning | ______ |

## Result

Thus, a pre-trained **VGG16 model from Keras** was successfully used for Transfer Learning in image classification. The pre-trained ImageNet features were reused and a new classifier was trained for the Cats vs Dogs classification task.

## Viva Questions

1. What is a pre-trained model?
2. What is Transfer Learning?
3. Why is VGG16 called a pre-trained model?
4. What is ImageNet?
5. What does `include_top=False` do?
6. Why do we freeze the VGG16 layers initially?
7. What is feature extraction?
8. What is fine-tuning?
9. Why is a small learning rate used during fine-tuning?
10. Name two other pre-trained models available in Keras.